install pacakages

In [ ]:
!pip -q install datasets pandas numpy scikit-learn

!apt-get -qq update
!apt-get -qq install -y ncbi-blast+

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ncbi-data.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Setting up ncbi-data (6.1.20170106+dfsg1-9) ...
Setting up ncbi-blast+ (2.12.0+ds-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...


In [ ]:
# Check BLAST installation
!blastp -version
!makeblastdb -version

blastp: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08
makeblastdb: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


Imports and load the dataset

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# Load Hugging Face dataset
dataset = load_dataset("tattabio/ec_classification")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/512 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Train shape: (512, 3)
Test shape: (128, 3)


,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


Inspect columns

In [ ]:
print(train_df.columns.tolist())
print(test_df.columns.tolist())

print("Unique train labels:", train_df["Label"].nunique())
print("Unique test labels:", test_df["Label"].nunique())

['Entry', 'Label', 'Sequence']
['Entry', 'Label', 'Sequence']
Unique train labels: 128
Unique test labels: 128


Write train and test FASTA files

In [ ]:
train_fasta = "/content/train_sequences.fasta"
test_fasta = "/content/test_sequences.fasta"

def write_fasta_with_label(df, fasta_path, include_label=True):
    with open(fasta_path, "w") as f:
        for _, row in df.iterrows():
            entry = str(row["Entry"])
            label = str(row["Label"])
            seq = str(row["Sequence"]).strip().upper()

            # Keep the header simple for easy parsing later
            if include_label:
                header = f"{entry}|{label}"
            else:
                header = entry

            f.write(f">{header}\n")
            f.write(f"{seq}\n")

write_fasta_with_label(train_df, train_fasta, include_label=True)
write_fasta_with_label(test_df, test_fasta, include_label=True)

print("location:")
print(train_fasta)
print(test_fasta)

Wrote:
/content/train_sequences.fasta
/content/test_sequences.fasta


In [ ]:
# Preview the training FASTA
!head -n 6 /content/train_sequences.fasta

>Q9LQC0|1.14.14.18
MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHLVRRANEDQTLVVNVVAAAGEKPERRYPREPNGFVEEMRFVVMKIHPRDQVKEGKSDSNDLVSTWNFTIEGYLKFLVDSKLVFETLERIINESAIQAYAGLKNTGLERAENLSRDLEWFKEQGYEIPESMVPGKAYSQYLKNIAEKDPPAFICHFYNINFAHSAGGRMIGTKVAEKILDNKELEFYKWDGQLSELLQNVSEELNKVAELWTREEKNHCLEETEKSFKFYWEIFRYLLS
>A0AFT8|1.14.14.18
MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFELWHSKPEDKDYEEVVVTSKWESEEAQRNWVKSDSFKKAHGRTKDTREQREDRKGIVGNEIARFEVVHVQNPVTIEK
>P74133|1.14.14.18
MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLYYLYSALEAALRQHRDNEIISAIYFPELNRTDKLAEDLTYYYGPNWQQIIQPTPCAKIYVDRLKTIAASEPELLIAHCYTRYLGDLSGGQSLKNIIRSALQLPEGEGTAMYEFDSLPTPGDRRQFKEIYRDVLNSLPLDEATINRIVEEANYAFSLNREVMHDLEDLIKAAIGEHTFDLLTRQDRPGSTEARSTAGHPITLMVGE


Build a BLAST protein database from the training set

In [ ]:
blast_db_prefix = "/content/ec_train_blastdb"

cmd = [
    "makeblastdb",
    "-in", train_fasta,
    "-dbtype", "prot",
    "-out", blast_db_prefix,
    "-parse_seqids"
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)



Building a new DB, current time: 04/22/2026 04:24:43
New DB name:   /content/ec_train_blastdb
New DB title:  /content/train_sequences.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 512 sequences in 0.058409 seconds.






In [ ]:
# Check the database files
!ls -lh /content/ec_train_blastdb*

-rw-r--r-- 1 root root  48K Apr 22 04:24 /content/ec_train_blastdb.pdb
-rw-r--r-- 1 root root  27K Apr 22 04:24 /content/ec_train_blastdb.phr
-rw-r--r-- 1 root root 4.2K Apr 22 04:24 /content/ec_train_blastdb.pin
-rw-r--r-- 1 root root 2.1K Apr 22 04:24 /content/ec_train_blastdb.pog
-rw-r--r-- 1 root root  12K Apr 22 04:24 /content/ec_train_blastdb.pos
-rw-r--r-- 1 root root 6.1K Apr 22 04:24 /content/ec_train_blastdb.pot
-rw-r--r-- 1 root root 272K Apr 22 04:24 /content/ec_train_blastdb.psq
-rw-r--r-- 1 root root  16K Apr 22 04:24 /content/ec_train_blastdb.ptf
-rw-r--r-- 1 root root 2.1K Apr 22 04:24 /content/ec_train_blastdb.pto


Run real BLASTP: test queries against the training database

In [ ]:
blast_output = "/content/blast_results_top10.tsv"

blast_fields = "qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"

cmd = [
    "blastp",
    "-query", test_fasta,
    "-db", blast_db_prefix,
    "-out", blast_output,
    "-outfmt", f"6 {blast_fields}",
    "-max_target_seqs", "10",
    "-evalue", "1e-5",
    "-num_threads", "2"
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
# Peek at BLAST results
!head -n 10 /content/blast_results_top10.tsv

O46310|1.17.4.1	Q6R7K3|1.17.4.1	28.000	300	202	7	36	325	73	368	1.20e-26	105
O46310|1.17.4.1	Q9KFH7|1.17.4.1	20.805	298	214	8	49	328	44	337	2.63e-15	70.9
O46310|1.17.4.1	Q9CBQ2|1.17.4.1	21.545	246	176	7	57	293	37	274	1.28e-10	56.6
Q5UQV6|1.8.3.2	Q12284|1.8.3.2	32.558	86	55	1	7	92	82	164	1.83e-12	57.0
Q5UQV6|1.8.3.2	Q91FH7|1.8.3.2	32.710	107	58	3	1	99	4	104	2.77e-11	52.4
Q8IRW8|2.1.1.354	Q54HS3|2.1.1.354	49.650	143	69	2	2290	2431	1346	1486	4.10e-40	158
Q8IRW8|2.1.1.354	Q75D88|2.1.1.354	51.064	141	67	2	2293	2431	835	975	1.67e-38	152
Q8IRW8|2.1.1.354	Q84WW6|2.1.1.354	36.306	157	95	3	2275	2430	76	228	1.12e-20	92.8
Q6BTC8|2.1.1.360	Q6C4Y5|2.1.1.360	44.177	249	132	5	927	1172	243	487	5.10e-51	183
Q6BTC8|2.1.1.360	Q8TEK3|2.1.1.360	26.923	234	165	5	940	1171	96	325	4.27e-16	78.6


Load BLAST results into pandas

In [ ]:
columns = [
    "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
    "qstart", "qend", "sstart", "send", "evalue", "bitscore"
]

blast_df = pd.read_csv(blast_output, sep="\t", header=None, names=columns)
blast_df.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,O46310|1.17.4.1,Q6R7K3|1.17.4.1,28.000,300,202,7,36,325,73,368,1.200000e-26,105.0
1,O46310|1.17.4.1,Q9KFH7|1.17.4.1,20.805,298,214,8,49,328,44,337,2.630000e-15,70.9
2,O46310|1.17.4.1,Q9CBQ2|1.17.4.1,21.545,246,176,7,57,293,37,274,1.280000e-10,56.6
3,Q5UQV6|1.8.3.2,Q12284|1.8.3.2,32.558,86,55,1,7,92,82,164,1.830000e-12,57.0
4,Q5UQV6|1.8.3.2,Q91FH7|1.8.3.2,32.710,107,58,3,1,99,4,104,2.770000e-11,52.4


In [ ]:
print("BLAST result rows:", len(blast_df))
print("Number of unique queries with hits:", blast_df["qseqid"].nunique())

BLAST result rows: 216
Number of unique queries with hits: 71


Parse labels from FASTA headers

Our FASTA headers were written as:

query: Entry|Label
subject: Entry|Label

In [ ]:
def parse_entry_and_label(header):
    parts = str(header).split("|")
    entry = parts[0]
    label = parts[1] if len(parts) > 1 else None
    return entry, label

# Parse query and subject metadata
blast_df[["query_entry", "query_label"]] = blast_df["qseqid"].apply(
    lambda x: pd.Series(parse_entry_and_label(x))
)
blast_df[["subject_entry", "subject_label"]] = blast_df["sseqid"].apply(
    lambda x: pd.Series(parse_entry_and_label(x))
)

blast_df.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,query_entry,query_label,subject_entry,subject_label
0,O46310|1.17.4.1,Q6R7K3|1.17.4.1,28.000,300,202,7,36,325,73,368,1.200000e-26,105.0,O46310,1.17.4.1,Q6R7K3,1.17.4.1
1,O46310|1.17.4.1,Q9KFH7|1.17.4.1,20.805,298,214,8,49,328,44,337,2.630000e-15,70.9,O46310,1.17.4.1,Q9KFH7,1.17.4.1
2,O46310|1.17.4.1,Q9CBQ2|1.17.4.1,21.545,246,176,7,57,293,37,274,1.280000e-10,56.6,O46310,1.17.4.1,Q9CBQ2,1.17.4.1
3,Q5UQV6|1.8.3.2,Q12284|1.8.3.2,32.558,86,55,1,7,92,82,164,1.830000e-12,57.0,Q5UQV6,1.8.3.2,Q12284,1.8.3.2
4,Q5UQV6|1.8.3.2,Q91FH7|1.8.3.2,32.710,107,58,3,1,99,4,104,2.770000e-11,52.4,Q5UQV6,1.8.3.2,Q91FH7,1.8.3.2


Top-1 BLAST baseline

In [ ]:
# Sort so the best hit for each query appears first
blast_top1 = (
    blast_df.sort_values(["qseqid", "bitscore", "pident"], ascending=[True, False, False])
            .drop_duplicates(subset=["qseqid"], keep="first")
            .reset_index(drop=True)
)

blast_top1.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,query_entry,query_label,subject_entry,subject_label
0,A0L6H5|3.1.11.6,A5I308|3.1.11.6,44.444,54,30,0,5,58,4,57,5.950000e-12,50.4,A0L6H5,3.1.11.6,A5I308,3.1.11.6
1,A0LL58|2.1.1.77,A5W820|2.1.1.77,50.244,205,99,3,5,207,4,207,9.110000e-65,194.0,A0LL58,2.1.1.77,A5W820,2.1.1.77
2,A1VYL9|3.6.1.9,Q11A17|3.6.1.9,25.000,188,135,2,2,183,6,193,2.130000e-12,58.2,A1VYL9,3.6.1.9,Q11A17,3.6.1.9
3,A1WYU0|6.3.2.4,Q2RJD5|6.3.2.4,40.850,306,167,4,12,303,2,307,6.420000e-65,202.0,A1WYU0,6.3.2.4,Q2RJD5,6.3.2.4
4,A3MTK6|3.1.12.1,Q97TX9|3.1.12.1,29.310,174,99,8,58,209,26,197,5.010000e-08,46.6,A3MTK6,3.1.12.1,Q97TX9,3.1.12.1


In [ ]:
top1_results = blast_top1[[
    "query_entry", "query_label",
    "subject_entry", "subject_label",
    "pident", "bitscore", "evalue"
]].copy()

top1_results["predicted_label"] = top1_results["subject_label"]
top1_results["correct"] = top1_results["query_label"] == top1_results["predicted_label"]

top1_results.head(20)

,query_entry,query_label,subject_entry,subject_label,pident,bitscore,evalue,predicted_label,correct
0,A0L6H5,3.1.11.6,A5I308,3.1.11.6,44.444,50.4,5.950000e-12,3.1.11.6,True
1,A0LL58,2.1.1.77,A5W820,2.1.1.77,50.244,194.0,9.110000e-65,2.1.1.77,True
2,A1VYL9,3.6.1.9,Q11A17,3.6.1.9,25.000,58.2,2.130000e-12,3.6.1.9,True
3,A1WYU0,6.3.2.4,Q2RJD5,6.3.2.4,40.850,202.0,6.420000e-65,6.3.2.4,True
4,A3MTK6,3.1.12.1,Q97TX9,3.1.12.1,29.310,46.6,5.010000e-08,3.1.12.1,True
5,A5FB63,3.2.1.14,Q13231,3.2.1.14,26.593,114.0,6.800000e-28,3.2.1.14,True
6,A8AC29,7.1.2.2,Q21Z99,7.1.2.2,28.022,73.9,2.070000e-15,7.1.2.2,True
7,A9BJK2,6.3.4.19,Q31G65,6.3.4.19,30.043,103.0,1.430000e-26,6.3.4.19,True
8,B1AJ41,2.8.1.13,Q8CXQ3,2.8.1.13,51.351,354.0,2.050000e-122,2.8.1.13,True
9,B4HEM4,3.1.4.35,Q9I7S6,3.1.4.35,38.028,114.0,3.460000e-27,3.1.4.35,True


evaluate

In [ ]:
y_true = top1_results["query_label"].tolist()
y_pred = top1_results["predicted_label"].tolist()

top1_accuracy = accuracy_score(y_true, y_pred)
top1_macro_f1 = f1_score(y_true, y_pred, average="macro")
top1_weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print(f"Top-1 BLAST Accuracy: {top1_accuracy:.4f}")
print(f"Top-1 BLAST Macro F1: {top1_macro_f1:.4f}")
print(f"Top-1 BLAST Weighted F1: {top1_weighted_f1:.4f}")

Top-1 BLAST Accuracy: 0.9718
Top-1 BLAST Macro F1: 0.9537
Top-1 BLAST Weighted F1: 0.9671


In [ ]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    1.17.4.1       1.00      1.00      1.00         1
     1.8.3.2       1.00      1.00      1.00         1
   2.1.1.354       1.00      1.00      1.00         1
   2.1.1.360       1.00      1.00      1.00         1
    2.1.1.37       1.00      1.00      1.00         1
    2.1.1.77       1.00      1.00      1.00         1
   2.3.1.225       1.00      1.00      1.00         1
   2.3.1.269       1.00      1.00      1.00         1
    2.3.1.48       1.00      1.00      1.00         1
    2.3.1.51       1.00      1.00      1.00         1
    2.3.2.23       1.00      1.00      1.00         1
    2.3.2.26       1.00      1.00      1.00         1
    2.3.2.27       0.00      0.00      0.00         0
    2.3.2.31       1.00      1.00      1.00         1
   2.4.1.109       1.00      1.00      1.00         1
   2.7.1.107       1.00      1.00      1.00         1
    2.7.1.21       1.00      1.00      1.00         1
    2.7.1.33       1.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

Handle any no-hit queries

In [ ]:
all_test_headers = [f"{row.Entry}|{row.Label}" for _, row in test_df.iterrows()]
predicted_headers = set(top1_results["query_entry"] + "|" + top1_results["query_label"])

missing_queries = [q for q in all_test_headers if q not in predicted_headers]
print("Queries with no BLAST hits:", len(missing_queries))
missing_queries[:10]

Queries with no BLAST hits: 57


['A0A0H2XEA6|1.14.14.18',
 'Q5AD07|1.15.1.1',
 'O74831|1.16.3.1',
 'Q8PU58|1.5.98.3',
 'Q09956|2.1.1.72',
 'Q6LWZ3|2.1.1.86',
 'A2BQ15|2.1.3.15',
 'Q810L3|2.3.2.27',
 'O64792|2.4.1.198',
 'P39421|2.4.2.31']